# Walkthrough: UCI Heart Disease pipeline

This notebook runs the same five steps as `run_all.py`, one at a time, so you can see the intermediate tables and figures. The code lives in `src/`; the notebook only calls it.

If you just want to check that your results match mine, `python run_all.py` is enough. Use this notebook when you want to look inside each step.

Run it from the repository root with the `dpbs-midterm` kernel.

In [ ]:
import json
import platform

import pandas as pd
import sklearn
from IPython.display import Image, display

from src import baseline, download, eda, harmonize, preprocess
from src.config import EXPECTED_DIR, MAX_MISSING_SHARE, RESULTS_DIR, SEED

pd.set_option("display.max_columns", 30)
print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Seed: {SEED} | drop threshold: {MAX_MISSING_SHARE:.0%} missing")

## Step 1. Raw files

The four UCI files in `data/raw/`. If one is missing, it gets downloaded. Compare the hashes with checkpoint C1 in the README.

In [ ]:
raw_info = download.run()
pd.DataFrame(raw_info).T

## Step 2. Load and harmonize

`?` becomes missing, and so do `chol = 0` and `trestbps = 0`, because nobody has a cholesterol or blood pressure of zero. Expect 920 rows.

In [ ]:
raw, df, zeros = harmonize.run()
print(f"Rows: {len(df)}")
print("Zeros turned into missing:", zeros)
df.head()

In [ ]:
# Missing counts before and after the zero rule
cols = [c for c in df.columns if c not in ("site", "num", "target")]
pd.DataFrame({
    "before (only '?')": raw[cols].isna().sum(),
    "after zero rule": df[cols].isna().sum(),
})

## Step 3. Exploratory analysis

Everything per site and pooled. The pooled numbers alone hide how different the hospitals are.

In [ ]:
summary, miss, miss_vs_label = eda.run(df)
summary

In [ ]:
display(Image(RESULTS_DIR / "eda" / "F1_prevalence_by_site.png"))

Missing values follow the site. Cleveland is nearly complete; `ca`, `thal` and `slope` are mostly empty elsewhere, and Switzerland has no cholesterol at all.

In [ ]:
display(Image(RESULTS_DIR / "eda" / "F2_missing_heatmap.png"))
miss

Disease rate among patients with and without each value. Where the two columns differ a lot, the fact that a value is missing already says something about the label.

In [ ]:
miss_vs_label

In [ ]:
display(Image(RESULTS_DIR / "eda" / "F3_numeric_by_site.png"))
pd.read_csv(RESULTS_DIR / "eda" / "T3_numeric_by_site_median_iqr.csv", index_col=0)

In [ ]:
display(Image(RESULTS_DIR / "eda" / "F4_correlation.png"))

## Step 4. Split, select, impute, encode

Split first (80/20, stratified by site and target). Everything after that is learned from the training rows only. Features with more than 30% missing in the training set are dropped.

In [ ]:
prep = preprocess.run(df)
print(f"Train {len(prep['train'])} / test {len(prep['test'])}")
print("Dropped:", prep["dropped"])
print(f"Encoded features ({len(prep['encoded_features'])}):", prep["encoded_features"])
pd.Series(prep["missing_share_train"], name="missing share (train)").sort_values(ascending=False)

In [ ]:
# The split keeps the site mix and the disease rate similar in train and test
pd.concat({
    "train": prep["train"].groupby("site")["target"].agg(["size", "mean"]),
    "test": prep["test"].groupby("site")["target"].agg(["size", "mean"]),
}, axis=1).round(3)

## Step 5. Sanity-check models

**M1** is a plain logistic regression on the processed features. **M2** sees only which values were missing. If M2 does clearly better than 0.5 AUC, missingness alone carries the label, which is why the pipeline adds no missing-value indicators and leaves `site` out of the features.

In [ ]:
res = baseline.run(prep)
print(f"M1  CV AUC (train): {res['m1_cv_auc_mean']} +/- {res['m1_cv_auc_sd']}")
print(f"M1  test AUC: {res['m1_test_auc']}   accuracy: {res['m1_test_accuracy']}")
print(f"M2  missingness-only test AUC: {res['m2_missingness_only_test_auc']}")
pd.DataFrame(res["m1_test_by_site"]).set_index("site")

In [ ]:
pd.read_csv(RESULTS_DIR / "model" / "M1_coefficients.csv", index_col=0)

## Check against the reference run

Compares a few headline numbers with `expected/checkpoints.json`. For the full comparison of every checkpoint, run `python run_all.py` in the terminal.

In [ ]:
ref = json.loads((EXPECTED_DIR / "checkpoints.json").read_text())
rows = [("rows", len(df), ref["C2_rows_total"])]
rows += [(f"prevalence % {site}", pct, ref["C3_prevalence_pct"][site])
         for site, pct in summary["prevalence_pct"].items()]
rows += [
    ("dropped features", ", ".join(prep["dropped"]), ", ".join(ref["C4_dropped_features"])),
    ("M1 test AUC", res["m1_test_auc"], ref["C5_models"]["m1_test_auc"]),
    ("M2 test AUC", res["m2_missingness_only_test_auc"], ref["C5_models"]["m2_missingness_only_test_auc"]),
]
check = pd.DataFrame(rows, columns=["checkpoint", "yours", "expected"]).set_index("checkpoint")
check["match"] = check["yours"] == check["expected"]
check